In [1]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Charan@123"  
)

print(" Connected to MySQL!")
conn.close()

 Connected to MySQL!


In [2]:
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Charan@123"  
)
cursor = conn.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS bank_analytics")
cursor.execute("USE bank_analytics")

cursor.execute("""
CREATE TABLE IF NOT EXISTS branches (
    branch_id   INT PRIMARY KEY,
    branch_name VARCHAR(100),
    city        VARCHAR(50),
    state       VARCHAR(50)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INT PRIMARY KEY,
    name        VARCHAR(100),
    age         INT,
    gender      VARCHAR(10),
    city        VARCHAR(50),
    occupation  VARCHAR(50),
    join_date   DATE
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS accounts (
    account_id   INT PRIMARY KEY,
    customer_id  INT,
    account_type VARCHAR(50),
    branch_id    INT,
    balance      DECIMAL(12,2),
    opened_date  DATE,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (branch_id)   REFERENCES branches(branch_id)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS transactions (
    transaction_id   INT PRIMARY KEY,
    account_id       INT,
    transaction_date DATE,
    amount           DECIMAL(12,2),
    transaction_type VARCHAR(20),
    FOREIGN KEY (account_id) REFERENCES accounts(account_id)
)
""")

conn.commit()
print(" Database and 4 tables created!")

 Database and 4 tables created!


In [7]:
import random
from faker import Faker

fake = Faker('en_IN')
random.seed(42)

cursor.execute("USE bank_analytics")

# 👇 Clear existing data first
cursor.execute("SET FOREIGN_KEY_CHECKS = 0")
cursor.execute("TRUNCATE TABLE transactions")
cursor.execute("TRUNCATE TABLE accounts")
cursor.execute("TRUNCATE TABLE customers")
cursor.execute("TRUNCATE TABLE branches")
cursor.execute("SET FOREIGN_KEY_CHECKS = 1")

cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad", "Pune", "Kolkata", "Ahmedabad"]
occupations = ["Engineer", "Doctor", "Teacher", "Businessman", "Student", "Lawyer", "Accountant", "Nurse"]
account_types = ["Savings", "Current", "Fixed Deposit", "Recurring Deposit"]

branches = [
    (1, "MG Road Branch",     "Bangalore",  "Karnataka"),
    (2, "Connaught Place",    "Delhi",      "Delhi"),
    (3, "Bandra Branch",      "Mumbai",     "Maharashtra"),
    (4, "Anna Nagar Branch",  "Chennai",    "Tamil Nadu"),
    (5, "Hitech City Branch", "Hyderabad",  "Telangana"),
]

cursor.executemany("INSERT INTO branches (branch_id, branch_name, city, state) VALUES (%s,%s,%s,%s)", branches)

customers = [(i, fake.name(), random.randint(18,70), random.choice(["Male","Female"]),
              random.choice(cities), random.choice(occupations),
              fake.date_between(start_date="-5y", end_date="-6m")) for i in range(1, 1001)]

cursor.executemany("INSERT INTO customers (customer_id,name,age,gender,city,occupation,join_date) VALUES (%s,%s,%s,%s,%s,%s,%s)", customers)

accounts = [(i, i, random.choice(account_types), random.randint(1,5),
             round(random.uniform(1000,500000),2),
             fake.date_between(start_date="-5y", end_date="-3m")) for i in range(1, 1001)]

cursor.executemany("INSERT INTO accounts (account_id,customer_id,account_type,branch_id,balance,opened_date) VALUES (%s,%s,%s,%s,%s,%s)", accounts)

transactions = [(i, random.randint(1,1000),
                 fake.date_between(start_date="-1y", end_date="today"),
                 round(random.uniform(100,50000),2),
                 random.choice(["Deposit","Withdrawal"])) for i in range(1, 5001)]

cursor.executemany("INSERT INTO transactions (transaction_id,account_id,transaction_date,amount,transaction_type) VALUES (%s,%s,%s,%s,%s)", transactions)

conn.commit()
print("Data inserted! 1000 customers, 5000 transactions.")

Data inserted! 1000 customers, 5000 transactions.


In [8]:
tables = ["branches", "customers", "accounts", "transactions"]

for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"{table}: {count} rows")

branches: 5 rows
customers: 1000 rows
accounts: 1000 rows
transactions: 5000 rows


In [4]:
import pandas as pd

# Read all 4 tables
branches = pd.read_sql("SELECT * FROM branches", conn)
customers = pd.read_sql("SELECT * FROM customers", conn)
accounts = pd.read_sql("SELECT * FROM accounts", conn)
transactions = pd.read_sql("SELECT * FROM transactions", conn)

# Save as CSV
branches.to_csv(r"C:\my project\finanace projects\bank_analytics\data\branches.csv", index=False)
customers.to_csv(r"C:\my project\finanace projects\bank_analytics\data\customers.csv", index=False)
accounts.to_csv(r"C:\my project\finanace projects\bank_analytics\data\accounts.csv", index=False)
transactions.to_csv(r"C:\my project\finanace projects\bank_analytics\data\transactions.csv", index=False)

print(" All 4 tables exported to CSV!")
print(f"Branches: {len(branches)} rows")
print(f"Customers: {len(customers)} rows")
print(f"Accounts: {len(accounts)} rows")
print(f"Transactions: {len(transactions)} rows")

 All 4 tables exported to CSV!
Branches: 5 rows
Customers: 1000 rows
Accounts: 1000 rows
Transactions: 5000 rows


C:\Users\USER\AppData\Local\Temp\ipykernel_8772\3409853809.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  branches = pd.read_sql("SELECT * FROM branches", conn)
C:\Users\USER\AppData\Local\Temp\ipykernel_8772\3409853809.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customers = pd.read_sql("SELECT * FROM customers", conn)
C:\Users\USER\AppData\Local\Temp\ipykernel_8772\3409853809.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  accounts = pd.read_sql("SELECT * FROM accounts", conn)
C:\Users\USER\AppData\Local\T